In [31]:
%run maths.ipynb
%run shell_mesh_builder.ipynb
%run chamber_septa_builder.ipynb
%run siphuncle_builder.ipynb
%run mesh_plotter.ipynb
%run animation_frames.ipynb

In [32]:
from pathlib import Path
import os
import json
import numpy as np

In [33]:
def load_options(options_file):
    """
    Load a JSON-formatted options file that acts as a "shell preset".
    
    The options file keeps the editable model parameters separate from the mesh-building
    code, making it easier to define different shell forms such as Nautilus-like,
    ammonite-like or other exploratory morphologies.

    :param options_file: Path to the options file
    :return: Dictionary of values read from the options file
    """
    # Load the options file.
    with open(options_file) as f:
        return json.load(f)

# A Note on Orthocones

Orthocones do not follow a logarithmic spiral, but the chamber/septa builder expects a 1D coordinate that increases along shell growth and
is used to determine septum placement.

For coiled shells this coordinate is naturally theta (growth angle), since shell development progresses around the spiral.

For orthocones there is no meaningful spiral angle. Instead we use the shell's axial growth coordinate (x) as the chamber-placement axis.

This means:

    theta = x

is intentional.

The resulting chamber_spacing parameter is interpreted as distance along the shell axis rather than angular spacing around a spiral.

Note that growth_phase is still generated separately because other parts of the shell-building pipeline use it for growth-dependent
calculations and pigmentation patterns.

In short:

    theta        -> chamber placement coordinate
    growth_phase -> synthetic growth progression parameter

These happen to be different for orthocones.

In [ ]:
def build_shell_components(options):
    """
    Build the 3D meshes for the shell, chamber septa and siphuncle from an options file.

    The JSON options file acts as a shell preset. It keeps the editable model
    param

    :param options: Dictionary of options including mesh building options
    :return: Tuple of the 3D mesh coordinates for the shell, chamber septa and siphuncle
    """
    growth_phase = None
    orientation_path = None
    z_path = None

    # Get the growth options that define the growth path followed by the shell aperture
    growth_options = options["growth"]
    shell_options = options["shell"]
    if growth_options["mode"] == "log_spiral":
        if growth_options["aperture_mode"] == "abutting":
            theta, r, x, y, width, height = logarithmic_spiral(
                a=growth_options["a"],
                b=growth_options["b"],
                turns=growth_options["turns"],
                points=growth_options["points"],
                aperture_mode="abutting",
                abutment_scale=growth_options["abutment_scale"],
                abutment_height_ratio=growth_options["abutment_height_ratio"]
            )
        else:
            theta, r, x, y, width, height = logarithmic_spiral(
                a=growth_options["a"],
                b=growth_options["b"],
                turns=growth_options["turns"],
                points=growth_options["points"],
                aperture_mode="proportional",
                aperture_width_multiplier=growth_options["aperture_width_multiplier"],
                aperture_height_multiplier=growth_options["aperture_height_multiplier"]
            )

    elif growth_options["mode"] == "conical_helix":
        theta, r, x, y, z_path, width, height = conical_helix(
            a=growth_options["a"],
            b=growth_options["b"],
            turns=growth_options["turns"],
            points=growth_options["points"],
            spire_height=growth_options["spire_height"],
            cone_taper=growth_options["cone_taper"],
            aperture_width_multiplier=growth_options["aperture_width_multiplier"],
            aperture_height_multiplier=growth_options["aperture_height_multiplier"]
        )

    elif growth_options["mode"] == "orthocone":
        # Note that x and z_path are intentionally switched as what's returned as "x"
        # will be 0 for an orthocone and z will be the growth axis
        t, r, z_path, y, x, width, height = centreline_axis(
            length=growth_options["length"],
            points=growth_options["points"],
            initial_radius=growth_options["initial_radius"],
            growth_rate=growth_options["growth_rate"],
            curvature=growth_options["curvature"],
            aperture_width_multiplier=growth_options["aperture_width_multiplier"],
            aperture_height_multiplier=growth_options["aperture_height_multiplier"],
            taper_mode=growth_options["taper_mode"],
        )

        # For orthocones there is no meaningful spiral angle. Instead we use the shell's axial growth
        # coordinate (x) as the chamber-placement axis. See the "Note on Orthocones", above
        theta = x
        growth_phase = np.linspace(0, 8 * np.pi, len(t))
        orientation_path = np.full(len(t), np.pi / 2)

    elif growth_options["mode"] == "cyrtocone":
        t, r, z_path, y, x, width, height = centreline_axis(
            length=growth_options["length"],
            points=growth_options["points"],
            initial_radius=growth_options["initial_radius"],
            growth_rate=growth_options["growth_rate"],
            curvature=growth_options["curvature"],
            aperture_width_multiplier=growth_options["aperture_width_multiplier"],
            aperture_height_multiplier=growth_options["aperture_height_multiplier"],
            taper_mode=growth_options["taper_mode"],
        )

        theta = x
        growth_phase = np.linspace(0, 8 * np.pi, len(t))

        dz = np.gradient(z_path)
        dy = np.gradient(y)
        orientation_path = np.arctan2(dy, dz)

    if shell_options["automatic_z_path"]:
        # ---------------------------------------------------------------------
        # Optional automatic Z path for whorl abutment
        # ---------------------------------------------------------------------
        # Derive vertical rise from aperture height. This helps high-spired
        # gastropods form touching/overlapping whorls rather than separated
        # screw-like coils.

        # whorl_abutment < 1.0  : Slight overlap between whorls
        # whorl_abutment <= 1.0 : Whorls just touching
        # whorl_abutment > 1.0  : Gaps between whorls
        pitch_per_turn = height * shell_options["whorl_abutment"]

        dtheta = np.gradient(theta)
        dz = (pitch_per_turn / (2 * np.pi)) * dtheta

        z_path = np.cumsum(dz)

        # Start at zero for tidier positioning
        z_path -= z_path[0]

    # Generate the 3D shell mesh.
    # This builds the outer shell surface, optional inner wall surface,
    # pigmentation, growth ribs and final aperture lip.
    shell_mesh = build_shell_mesh(
        theta,
        r,
        x,
        y,
        width,
        height,
        z_scale=shell_options["z_scale"],
        z_path=z_path,
        orientation_path=orientation_path,
        growth_phase=growth_phase,
        aperture_points=shell_options["aperture_points"],
        aperture_tilt=shell_options["aperture_tilt"],
        ribbing_amplitude=shell_options["ribbing_amplitude"],
        ribbing_frequency=shell_options["ribbing_frequency"],
        lip_start=shell_options["lip_start"],
        lip_strength=shell_options["lip_strength"],
        lip_power=shell_options["lip_power"],
        wall_enabled=shell_options["wall_enabled"],
        wall_thickness=shell_options["wall_thickness"],
        inner_surface_colour_shift=shell_options["inner_surface_colour_shift"],
        use_centreline_frame=growth_options["mode"] in ["orthocone", "cyrtocone"]
    )

    # Generate the septa separating the chambers.
    # These are the curved internal walls left behind as the shell grows.
    chamber_options = options["chambers"]
    if chamber_options["enabled"]:
        chamber_mesh = build_chamber_septa(
            theta,
            shell_mesh,
            aperture_points=shell_options["aperture_points"],
            chamber_spacing=chamber_options["chamber_spacing"],
            septum_rings=chamber_options["septum_rings"],
            septum_depth=chamber_options["septum_depth"]
        )
    else:
        chamber_mesh = None

    # Generate the siphuncle.
    # The siphuncle is tied to the chambered region of the shell. By default,
    # it uses the same spacing as the chamber septa and stops near the start
    # of the final body chamber / lip region, rather than running all the way
    # to the shell mouth.
    siphuncle_options = options["siphuncle"]
    if siphuncle_options["enabled"]:
        siphuncle_mesh = build_siphuncle_mesh(
            theta,
            shell_mesh,
            aperture_points=shell_options["aperture_points"],
            siphuncle_radius=siphuncle_options["siphuncle_radius"],
            siphuncle_offset=siphuncle_options["siphuncle_offset"],
            tube_points=siphuncle_options["tube_points"],
            chamber_spacing=chamber_options["chamber_spacing"],
            siphuncle_end_fraction=siphuncle_options.get(
                "siphuncle_end_fraction",
                shell_options.get("lip_start", 0.90)
            ),
            siphuncle_taper_enabled=siphuncle_options["siphuncle_taper_enabled"],
            siphuncle_taper_power=siphuncle_options["siphuncle_taper_power"],
            siphuncle_min_radius=siphuncle_options["siphuncle_min_radius"],
            siphuncle_max_radius=siphuncle_options["siphuncle_max_radius"],
            include_terminal_ring=siphuncle_options["include_terminal_ring"],
            fixed_offset_angle=siphuncle_options["fixed_offset_angle"],
            growth_mode=growth_options["mode"]
        )
    else:
        siphuncle_mesh = None

    return shell_mesh, chamber_mesh, siphuncle_mesh


In [35]:
def save_views(
    name,
    opaque_plotting_options,
    transparent_plotting_options,
    shell_mesh,
    chamber_mesh,
    siphuncle_mesh,
    views):
    """
    Render and save each of the specified views for the specified shell meshes

    :param opaque_plotting_options: Plotting options for the opaque shell view
    :param transparent_plotting_options: Plotting options for the transparent shell view
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    :param views: Named views to render
    """

    # Set the export folder path
    export_folder = Path(os.getcwd()).parent / "renders"

    # Iterate over the viewpoints
    for view, zoom in views.items():
        # Plot the shell with an opaque surface
        if opaque_plotting_options["enabled"]:
            plotting_options = {**opaque_plotting_options, "hide_axes": True}
            png_path = export_folder / f"{name}-opaque-{view}.png"
            plot_and_save_shell_mesh(plotting_options, shell_mesh, png_path, chamber_mesh, siphuncle_mesh, view, zoom)
            
        # Plot the shell with a trasnsparent surface
        if transparent_plotting_options["enabled"]:
            plotting_options = {**transparent_plotting_options, "hide_axes": True}
            png_path = export_folder / f"{name}-transparent-{view}.png"
            plot_and_save_shell_mesh(plotting_options, shell_mesh, png_path, chamber_mesh, siphuncle_mesh, view, zoom)

In [36]:
def render_interactive_views(
    opaque_plotting_options,
    transparent_plotting_options,
    shell_mesh,
    chamber_mesh,
    siphuncle_mesh):
    """
    Render the interactive shell views

    :param opaque_plotting_options: Plotting options for the opaque shell view
    :param transparent_plotting_options: Plotting options for the transparent shell view
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    """

    # Plot the shell with an opaque surface
    if opaque_plotting_options["enabled"]:
        plot_shell_mesh(
            opaque_plotting_options,
            shell_mesh,
            chamber_mesh,
            siphuncle_mesh,
            opaque_plotting_options["viewpoint"],
            opaque_plotting_options["zoom"])

    # Plot the shell with a transparent surface to reveal internal structure
    if transparent_plotting_options["enabled"]:
        plot_shell_mesh(
            transparent_plotting_options,
            shell_mesh,
            chamber_mesh,
            siphuncle_mesh,
            opaque_plotting_options["viewpoint"],
            opaque_plotting_options["zoom"])

In [37]:
def render_interactive(options_file_name):
    """
    Render a 3D shell and render in-notebook interactive views of it

    :param options_file_name: Name of the options file to use, excluding path and extension
    """

    # Load the options file
    options_file_path = Path(os.getcwd()).parent / "data" / f"{options_file_name}.json"
    options = load_options(options_file_path)

    # Build the meshes describing the shell's features
    shell_mesh, chamber_mesh, siphuncle_mesh = build_shell_components(options)

    # Extract the plotting options
    opaque_plotting_options = options["plotting"]["opaque"]
    transparent_plotting_options = options["plotting"]["transparent"]

    # Render the interactive views, if required
    render_interactive_views(
        opaque_plotting_options,
        transparent_plotting_options,
        shell_mesh,
        chamber_mesh,
        siphuncle_mesh)

In [38]:
def render_static(options_file_name, views):
    """
    Render a 3D shell and save static views of it from the specified set of viewpoints

    :param options_file_name: Name of the options file to use, excluding path and extension
    :param views: List of viewpoints to save as PNG files
    """

    # Load the options file
    options_file_path = Path(os.getcwd()).parent / "data" / f"{options_file_name}.json"
    options = load_options(options_file_path)

    # Build the meshes describing the shell's features
    shell_mesh, chamber_mesh, siphuncle_mesh = build_shell_components(options)

    # Extract the plotting options
    opaque_plotting_options = options["plotting"]["opaque"]
    transparent_plotting_options = options["plotting"]["transparent"]

    # Save the specified views
    save_views(
        options_file_name,
        opaque_plotting_options,
        transparent_plotting_options,
        shell_mesh,
        chamber_mesh,
        siphuncle_mesh,
        views)

In [ ]:
def render_animation(options_file_name, render_type, view, zoom):
    """
    Render a 3D shell and save static views of it from the specified set of viewpoints

    :param options_file_name: Name of the options file to use, excluding path and extension
    :param render_type: One of "opaque" or "transparent"
    :param view: The viewpoint to view the animation from
    :param zoom: Zoom factor (camera distance); smaller zooms in
    """

    # Load the options file
    options_file_path = Path(os.getcwd()).parent / "data" / f"{options_file_name}.json"
    options = load_options(options_file_path)

    # Build the meshes describing the shell's features
    shell_mesh, chamber_mesh, siphuncle_mesh = build_shell_components(options)

    # Extract the plotting options
    plotting_options = {**options["plotting"][render_type], "hide_axes": True}

    # Render the animation frames
    output_stem = f"{options_file_name}-{render_type}-{view}"
    frames_folder_path = Path(os.getcwd()).parent / "frames" / output_stem
    _ = render_growth_animation_frames(
        plotting_options,
        shell_mesh,
        frames_folder_path,
        chamber_mesh,
        siphuncle_mesh,
        view,
        zoom)

    # Render the final movie
    movie_file_path = Path(os.getcwd()).parent / "movies" / f"{output_stem}.mp4"
    render_movie_from_frames(frames_folder_path, movie_file_path)
